In [3]:
from manim import *
import numpy as np
import math

class GaussianDistributionExplained(Scene):
    def construct(self):
        # --- Layout Setup ---
        # Vertical separator line
        separator = Line(UP * 4, DOWN * 4, stroke_width=2, color=GRAY)
        separator.shift(LEFT * 2) 
        
        # Define areas
        text_center = LEFT * 4.5
        anim_center = RIGHT * 2.5
        
        self.play(Create(separator))
        
        # --- PART 1: THE ANATOMY OF THE FORMULA ---
        # Increased font size
        header = Text("The Gaussian Anatomy", font_size=36, color=BLUE).move_to(text_center + UP * 3.5)
        
        # Formula: Increased font size
        formula = MathTex(
            r"f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}",
            font_size=32
        ).next_to(header, DOWN, buff=0.4)
        
        # Detailed dissection text: Increased font size and spacing
        anatomy_list = VGroup(
            Text("1. Normalizer:", font_size=24, color=ORANGE),
            MathTex(r"\frac{1}{\sigma\sqrt{2\pi}} \rightarrow \text{Ensures Area = 1}", font_size=24),
            Text("2. Position:", font_size=24, color=TEAL),
            MathTex(r"(x - \mu) \rightarrow \text{Shifts Center}", font_size=24),
            Text("3. Decay Shape:", font_size=24, color=YELLOW),
            MathTex(r"e^{-z^2} \rightarrow \text{Creates the Bell}", font_size=24)
        ).arrange(DOWN, center=False, aligned_edge=LEFT, buff=0.25).next_to(formula, DOWN, buff=0.5)
        
        self.play(Write(header), Write(formula))
        self.play(FadeIn(anatomy_list, lag_ratio=0.1))
        
        # --- GRAPH SETUP ---
        # NOTE: Y-range increased to 1.1 to accommodate Part 5 scenarios
        ax = Axes(
            x_range=[-4, 4, 1],
            y_range=[0, 1.1, 0.1], 
            x_length=6, y_length=4,
            axis_config={"include_tip": True, "tip_shape": StealthTip},
            tips=False
        ).move_to(anim_center + DOWN * 0.5)
        
        # Axis labels
        x_lbl = ax.get_x_axis_label(Text("Value (x)", font_size=20), direction=DOWN, buff=0.5)
        y_lbl = ax.get_y_axis_label(Text("Density (y)", font_size=20).rotate(90*DEGREES), direction=LEFT, buff=0.8)
        
        self.play(Create(ax), Write(x_lbl), Write(y_lbl))
        
        # Standard Normal Function
        def get_pdf(x, mu=0, sigma=1):
            return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

        def get_cdf(x, mu=0, sigma=1):
            return 0.5 * (1 + math.erf((x - mu) / (sigma * math.sqrt(2))))

        curve = ax.plot(lambda x: get_pdf(x), color=YELLOW)
        self.play(Create(curve), run_time=1.5)
        self.wait(1)

        # --- PART 2: SKEWNESS & CENTRAL TENDENCY CHECKS ---
        
        # Clear previous text
        self.play(FadeOut(anatomy_list), FadeOut(header), FadeOut(formula))
        
        skew_header = Text("Skewness Checks", font_size=36, color=RED_B).move_to(text_center + UP * 3.5)
        # Multi-line text for rules
        skew_rules = VGroup(
            Text("Visualizing the 'Tug of War'", font_size=24),
            Text("between Mean, Median, Mode.", font_size=24)
        ).arrange(DOWN, center=False, aligned_edge=LEFT, buff=0.1).next_to(skew_header, DOWN, buff=0.3)
        
        self.play(Write(skew_header), Write(skew_rules))

        # --- 2A. Right Skew Animation ---
        
        right_skew_curve = ax.plot(
            lambda x: get_pdf(x, -0.5, 0.7) * (1 if x < -0.5 else np.exp(-0.2*(x+0.5))),
            x_range=[-4, 4], color=RED
        )
        
        # Multi-line title
        rs_title = VGroup(
            Text("Right (Positive)", font_size=26, color=RED),
            Text("Skew", font_size=26, color=RED)
        ).arrange(DOWN, center=False, aligned_edge=LEFT, buff=0.1).next_to(skew_rules, DOWN, buff=0.5)
        
        rs_check = MathTex(r"\text{Mode} < \text{Median} < \text{Mean}", font_size=26, color=RED).next_to(rs_title, DOWN, buff=0.3)
        
        self.play(Transform(curve, right_skew_curve), FadeIn(rs_title), FadeIn(rs_check))
        
        # Lines and labels
        line_mode = ax.get_vertical_line(ax.c2p(-0.5, 0.55), color=WHITE)
        line_med = ax.get_vertical_line(ax.c2p(0.0, 0.45), color=YELLOW)
        line_mean = ax.get_vertical_line(ax.c2p(0.5, 0.35), color=TEAL)
        
        lbl_mode = Text("Mo", font_size=20, color=WHITE).next_to(line_mode, UP, buff=0.1)
        lbl_med = Text("Me", font_size=20, color=YELLOW).next_to(line_med, UP, buff=0.1)
        lbl_mean = Text("Mu", font_size=20, color=TEAL).next_to(line_mean, UP, buff=0.1).shift(DOWN*0.2)
        
        self.play(Create(line_mode), Write(lbl_mode))
        self.play(Create(line_med), Write(lbl_med))
        self.play(Create(line_mean), Write(lbl_mean))
        self.wait(2)
        
        # Clean up Right Skew
        self.play(
            FadeOut(rs_title), FadeOut(rs_check), 
            FadeOut(line_mode), FadeOut(line_med), FadeOut(line_mean),
            FadeOut(lbl_mode), FadeOut(lbl_med), FadeOut(lbl_mean)
        )

        # --- 2B. Left Skew Animation ---
        
        left_skew_curve = ax.plot(
            lambda x: get_pdf(x, 0.5, 0.7) * (1 if x > 0.5 else np.exp(0.2*(x-0.5))),
            x_range=[-4, 4], color=GREEN
        )
        
        # Multi-line title
        ls_title = VGroup(
             Text("Left (Negative)", font_size=26, color=GREEN),
             Text("Skew", font_size=26, color=GREEN)
        ).arrange(DOWN, center=False, aligned_edge=LEFT, buff=0.1).next_to(skew_rules, DOWN, buff=0.5)
        
        ls_check = MathTex(r"\text{Mean} < \text{Median} < \text{Mode}", font_size=26, color=GREEN).next_to(ls_title, DOWN, buff=0.3)
        
        self.play(Transform(curve, left_skew_curve), FadeIn(ls_title), FadeIn(ls_check))
        
        # Visual positions
        line_mean_L = ax.get_vertical_line(ax.c2p(-0.5, 0.35), color=TEAL)
        line_med_L = ax.get_vertical_line(ax.c2p(0.0, 0.45), color=YELLOW)
        line_mode_L = ax.get_vertical_line(ax.c2p(0.5, 0.55), color=WHITE)
        
        lbl_mean_L = Text("Mu", font_size=20, color=TEAL).next_to(line_mean_L, UP, buff=0.1).shift(DOWN*0.2)
        lbl_med_L = Text("Me", font_size=20, color=YELLOW).next_to(line_med_L, UP, buff=0.1)
        lbl_mode_L = Text("Mo", font_size=20, color=WHITE).next_to(line_mode_L, UP, buff=0.1)
        
        self.play(Create(line_mean_L), Write(lbl_mean_L))
        self.play(Create(line_med_L), Write(lbl_med_L))
        self.play(Create(line_mode_L), Write(lbl_mode_L))
        self.wait(2)
        
        # Clean up Skew
        self.play(
            FadeOut(skew_header), FadeOut(skew_rules), FadeOut(ls_title), FadeOut(ls_check),
            FadeOut(line_mean_L), FadeOut(line_med_L), FadeOut(line_mode_L),
            FadeOut(lbl_mean_L), FadeOut(lbl_med_L), FadeOut(lbl_mode_L)
        )
        
        # Back to Normal
        standard_curve = ax.plot(lambda x: get_pdf(x), color=YELLOW)
        self.play(Transform(curve, standard_curve))

        # --- PART 3: QUARTILES & BOX PLOT MEASURES ---
        
        stats_header = Text("Statistical Measures", font_size=36, color=BLUE).move_to(text_center + UP * 3.5)
        
        # Detailed list
        stats_list = VGroup(
            MathTex(r"\mu \text{ (Mean): Center}", color=TEAL, font_size=24),
            MathTex(r"Q_1: \text{Bottom 25\%}", color=PURPLE, font_size=24),
            MathTex(r"Q_2 \text{ (Median): 50\%}", color=TEAL, font_size=24),
            MathTex(r"Q_3: \text{Top 25\% Starts}", color=PURPLE, font_size=24),
            MathTex(r"\text{IQR}: Q_3 - Q_1", color=BLUE, font_size=24)
        ).arrange(DOWN, center=False, aligned_edge=LEFT, buff=0.3).next_to(stats_header, DOWN, buff=0.5)
        
        self.play(Write(stats_header), FadeIn(stats_list, lag_ratio=0.1))
        
        # Draw Q1, Q2, Q3
        q1_x, q2_x, q3_x = -0.675, 0, 0.675
        
        line_q2 = ax.get_vertical_line(ax.c2p(q2_x, get_pdf(q2_x)), color=TEAL)
        line_q1 = ax.get_vertical_line(ax.c2p(q1_x, get_pdf(q1_x)), color=PURPLE)
        line_q3 = ax.get_vertical_line(ax.c2p(q3_x, get_pdf(q3_x)), color=PURPLE)
        
        # Increased font for labels
        label_q1 = MathTex("Q_1", font_size=24, color=PURPLE).next_to(line_q1, DOWN, buff=0.3)
        label_q2 = MathTex(r"\mu", font_size=24, color=TEAL).next_to(line_q2, DOWN, buff=0.3)
        label_q3 = MathTex("Q_3", font_size=24, color=PURPLE).next_to(line_q3, DOWN, buff=0.3)
        
        self.play(Create(line_q2), Write(label_q2))
        self.play(Create(line_q1), Create(line_q3), Write(label_q1), Write(label_q3))
        
        # Highlight IQR area
        area_iqr = ax.get_area(curve, x_range=[q1_x, q3_x], color=BLUE, opacity=0.4)
        iqr_tag = Text("IQR (50%)", font_size=20, color=WHITE).move_to(ax.c2p(0, 0.2))
        
        self.play(FadeIn(area_iqr), Write(iqr_tag))
        self.wait(2)

        # --- PART 4: THE EMPIRICAL RULE (68-95-99.7) ---
        
        # Cleanup
        self.play(
            FadeOut(stats_header), FadeOut(stats_list), 
            FadeOut(line_q1), FadeOut(line_q2), FadeOut(line_q3),
            FadeOut(label_q1), FadeOut(label_q2), FadeOut(label_q3),
            FadeOut(area_iqr), FadeOut(iqr_tag)
        )
        
        emp_header = Text("Empirical Rule", font_size=36, color=GOLD).move_to(text_center + UP * 3.5)
        
        emp_details = VGroup(
            MathTex(r"1\sigma \text{ Range}: 68\%", color=YELLOW, font_size=24),
            MathTex(r"2\sigma \text{ Range}: 95\%", color=ORANGE, font_size=24),
            MathTex(r"3\sigma \text{ Range}: 99.7\%", color=RED, font_size=24)
        ).arrange(DOWN, center=False, aligned_edge=LEFT, buff=0.3).next_to(emp_header, DOWN, buff=0.5)
        
        self.play(Write(emp_header), Write(emp_details))
        
        # Y-position for arrows and labels
        arrow_y_pos = 0.15
        
        # 1 Sigma
        area_1s = ax.get_area(curve, x_range=[-1, 1], color=YELLOW, opacity=0.5)
        arrow_1s = DoubleArrow(start=ax.c2p(-1, arrow_y_pos), end=ax.c2p(1, arrow_y_pos), color=YELLOW, buff=0)
        lbl_1s = MathTex(r"68\%", font_size=24, color=YELLOW).next_to(arrow_1s, UP, buff=0.1)
        
        self.play(FadeIn(area_1s), Create(arrow_1s), Write(lbl_1s))
        self.wait(1)
        
        # 2 Sigma
        area_2s = ax.get_area(curve, x_range=[-2, 2], color=ORANGE, opacity=0.3)
        arrow_2s = DoubleArrow(start=ax.c2p(-2, arrow_y_pos), end=ax.c2p(2, arrow_y_pos), color=ORANGE, buff=0)
        lbl_2s = MathTex(r"95\%", font_size=24, color=ORANGE).next_to(arrow_2s, UP, buff=0.1)
        
        self.play(
            Transform(area_1s, area_2s), 
            ReplacementTransform(arrow_1s, arrow_2s), 
            ReplacementTransform(lbl_1s, lbl_2s)
        )
        self.wait(1)
        
        # 3 Sigma
        area_3s = ax.get_area(curve, x_range=[-3, 3], color=RED, opacity=0.2)
        arrow_3s = DoubleArrow(start=ax.c2p(-3, arrow_y_pos), end=ax.c2p(3, arrow_y_pos), color=RED, buff=0)
        lbl_3s = MathTex(r"99.7\%", font_size=24, color=RED).next_to(arrow_3s, UP, buff=0.1)
        
        self.play(
            Transform(area_1s, area_3s), 
            ReplacementTransform(arrow_2s, arrow_3s), 
            ReplacementTransform(lbl_2s, lbl_3s)
        )
        self.wait(2)
        
        # --- PART 5: 10 SCENARIOS ---
        
        # Cleanup Part 4
        self.play(FadeOut(emp_header), FadeOut(emp_details), FadeOut(area_1s), FadeOut(arrow_3s), FadeOut(lbl_3s))

        # Header
        scenarios_header = Text("10 Gaussian Scenarios", font_size=36, color=PURPLE).move_to(text_center + UP * 3.5)
        self.play(Write(scenarios_header))

        # Data Definitions for Scenarios
        scenarios = [
            {"id": "high_var", "title": "1. High Variance", "color": BLUE, "points": ["Large Sigma (σ=2.0)", "Spread out data", "Low peak"], "func": lambda x: get_pdf(x, 0, 2.0), "peaks": [0]},
            {"id": "low_var", "title": "2. Low Variance", "color": GREEN, "points": ["Small Sigma (σ=0.4)", "Clustered data", "High precision"], "func": lambda x: get_pdf(x, 0, 0.4), "peaks": [0]},
            {"id": "pos_shift", "title": "3. Positive Shift", "color": TEAL, "points": ["Mean (μ) = +2.0", "Moves Right"], "func": lambda x: get_pdf(x, 2, 1.0), "peaks": [2]},
            {"id": "neg_shift", "title": "4. Negative Shift", "color": MAROON, "points": ["Mean (μ) = -1.5", "Centered negative"], "func": lambda x: get_pdf(x, -1.5, 1.0), "peaks": [-1.5]},
            {"id": "bimodal", "title": "5. Bimodal", "color": ORANGE, "points": ["Two peaks", "Mixture model"], "func": lambda x: 0.5 * get_pdf(x, -1.5, 0.6) + 0.5 * get_pdf(x, 1.5, 0.6), "peaks": [-1.5, 1.5]},
            {"id": "ab_test", "title": "6. A/B Testing", "color": YELLOW, "points": ["Control vs Variant", "Overlap"], "custom_anim": True, "peaks": [-1, 1]},
            {"id": "clt", "title": "7. CLT (Sampling)", "color": GOLD, "points": ["N increases", "Curve narrows"], "func": lambda x: get_pdf(x, 0, 0.3), "peaks": [0]},
            {"id": "fat_tail", "title": "8. Fat Tails", "color": RED, "points": ["Black Swans", "Thicker tails"], "func": lambda x: get_pdf(x, 0, 0.5) + (0.05 if abs(x) > 1 else 0), "peaks": [0]},
            {"id": "cdf", "title": "9. CDF", "color": PURPLE, "points": ["Cumulative", "S-Curve"], "func": lambda x: get_cdf(x, 0, 1), "peaks": [0]},
            {"id": "zscore", "title": "10. Z-Score", "color": WHITE, "points": ["Z=2.5", "Outlier"], "custom_anim": True, "peaks": [2.5]}
        ]

        current_text = VGroup()
        current_dots = VGroup()

        for data in scenarios:
            # 1. Text Update
            new_text = VGroup(
                Text(data["title"], font_size=24, color=data["color"]),
                *[Text(f"- {p}", font_size=20) for p in data["points"]]
            ).arrange(DOWN, aligned_edge=LEFT).next_to(scenarios_header, DOWN, buff=0.5)

            if current_text:
                self.play(FadeOut(current_text), FadeIn(new_text), run_time=0.6)
            else:
                self.play(FadeIn(new_text))
            current_text = new_text

            # 2. Graph Update
            self.play(FadeOut(current_dots), run_time=0.2)
            new_dots = VGroup()

            if data.get("custom_anim"):
                if data["id"] == "ab_test":
                    curve_b = ax.plot(lambda x: get_pdf(x, -1, 1), color=BLUE, stroke_opacity=0.5)
                    curve_v = ax.plot(lambda x: get_pdf(x, 1, 1), color=RED, stroke_opacity=0.5)
                    group_ab = VGroup(curve_b, curve_v)
                    new_dots.add(Dot(ax.c2p(-1, get_pdf(-1, -1, 1)), color=BLUE), Dot(ax.c2p(1, get_pdf(1, 1, 1)), color=RED))
                    
                    self.play(FadeOut(curve)) 
                    self.play(Create(group_ab), Create(new_dots))
                    self.wait(1.5)
                    self.play(FadeOut(group_ab), FadeIn(curve))
                    
                elif data["id"] == "zscore":
                    std_curve = ax.plot(lambda x: get_pdf(x), color=WHITE)
                    self.play(Transform(curve, std_curve))
                    z_val = data["peaks"][0]
                    z_line = ax.get_vertical_line(ax.c2p(z_val, get_pdf(z_val)), color=RED)
                    z_dot = Dot(ax.c2p(z_val, get_pdf(z_val)), color=RED)
                    z_lbl = MathTex("Z=2.5", color=RED).next_to(z_dot, UP)
                    new_dots.add(z_dot)
                    self.play(Create(z_line), Create(z_dot), Write(z_lbl))
                    self.wait(2)
                    self.play(FadeOut(z_line), FadeOut(z_lbl))

            else:
                func = data["func"]
                new_curve = ax.plot(func, color=data["color"])
                for x_val in data["peaks"]:
                    dot = Dot(ax.c2p(x_val, func(x_val)), color=(RED if data["color"] == WHITE else data["color"]))
                    new_dots.add(dot)
                self.play(Transform(curve, new_curve), Create(new_dots), run_time=1.2)
                self.wait(1.5)
            
            current_dots = new_dots

        # Final cleanup
        self.play(FadeOut(scenarios_header), FadeOut(current_text), FadeOut(curve), FadeOut(ax), FadeOut(current_dots))

%manim -qk -v warning GaussianDistributionExplained

Manim Community v0.19.0